# 📝 AI Quiz Generator: run and evaluate the pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/video-quiz-generator/training/notebook.ipynb)

Turns a recording (or a pasted transcript) into a multiple-choice quiz: PyAV decodes the audio, Whisper
transcribes it and Qwen2.5-Instruct writes the quiz as strict JSON, which is parsed leniently and validated
strictly (up to 2 repair turns, then one fresh sampled generation).
Nothing is trained here: every model is a pretrained checkpoint. This notebook runs the Space's own
`space/pipeline.py` over a small grid, measures the **schema-valid rate**, scores every question with an
LLM judge (G-Eval-lite) and exports the results to `training/outputs/`.

1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training (run the pipeline) · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / space code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "video-quiz-generator"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../space/pipeline.py").exists():   # space/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/space"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: src/ puts ../space on sys.path, so `pipeline` is the Space's pipeline.py
import time

import pandas as pd
from IPython.display import Audio, Image, JSON, display

import src  # noqa: F401
from src import config, data_setup, engine, export, model_builder, utils
import pipeline as P

print(utils.lib_versions("torch", "transformers", "av", "numpy", "pandas"))
print("device:", device, "| pipeline for this device:", P.ASR_MODELS[P._kind(device)], "+", P.LLM_MODELS[P._kind(device)])

## 2. Config
Every setting is in `src/config.py`. `SMOKE_TEST=1` (environment variable or the line below) runs one
variant on one sample with a 2-question quiz and the small judge, in about 2 minutes on a CPU; it writes
only to `outputs/smoke/`. Nothing is pushed to the Hub from here: there are no weights, and the Space is
deployed by committing `space/` (see the root `ADDING_A_PROJECT.md`).

In [ ]:
import os
os.environ.setdefault("SMOKE_TEST", "0")      # set to "1" for the quick sanity run
cfg = config.Config()
cfg

## 3. Data
Two inputs: the 11 s JFK inaugural clip (audio, goes through the ASR; its true text is known, so the
word error rate can be measured) and the Gettysburg Address (text, the "paste a transcript" path).
`jfk.wav` is cached in `training/data/` (downloaded only if missing), and `sample.mp4` is built from it
with PyAV so the video path can be tested too.

In [ ]:
samples = data_setup.load_samples(cfg.samples)
display(pd.DataFrame([{k: (v[:90] + "..." if isinstance(v, str) and len(v) > 90 else v) for k, v in s.items()}
                      for s in samples]))
audio, _ = P.load_audio(utils.DATA_DIR / "jfk.wav")
print(f"jfk.wav: {audio.size / P.SAMPLE_RATE:.2f} s at {P.SAMPLE_RATE} Hz")
display(Audio(audio, rate=P.SAMPLE_RATE))

## 4. Load model
`P.load(device)` builds exactly what the Space runs on this hardware (GPU: whisper-small + Qwen2.5-1.5B,
CPU: whisper-tiny + Qwen2.5-0.5B). The evaluation runs every variant in `cfg.variants`.

In [ ]:
pipe = P.load(device)
display(pd.DataFrame(model_builder.describe(pipe)))
print("variants to evaluate:", cfg.variants)

## 5. Training
There is nothing to train: all models are pretrained. This step runs the pipeline over the evaluation
grid instead (each variant: transcribe the audio samples, then one quiz per sample and difficulty),
reusing the loaded pipeline when it already is the variant, and freeing it before loading the next.

In [ ]:
utils.set_seeds(cfg.seed, "torch")
run_start = time.perf_counter()
runs, params = {}, {}
for name in cfg.variants:
    if not model_builder.matches(pipe, cfg, name):
        del pipe
        model_builder.release()
        pipe = model_builder.build_variant(cfg, name, device)
    params[name] = sum(row["params"] for row in model_builder.describe(pipe))
    runs[name] = engine.run_variant(pipe, name, samples, cfg)

In [ ]:
quiz_table = pd.DataFrame([{k: q[k] for k in ("variant", "sample", "difficulty", "n_delivered", "n_requested",
                                               "valid", "valid_first_try", "strict_json_first_try", "attempts",
                                               "repairs", "regenerated", "seconds")}
                            for run in runs.values() for q in run["quizzes"]])
display(quiz_table)
display(pd.DataFrame([{k: a[k] for k in ("variant", "sample", "asr_model", "wer", "seconds", "text")}
                      for run in runs.values() for a in run["asr"]]))
ax = quiz_table.assign(run=quiz_table["sample"] + "/" + quiz_table["difficulty"]).pivot(
    index="run", columns="variant", values="seconds").plot.bar(rot=0, figsize=(7, 3), title="seconds per quiz")
ax.set_ylabel("s");

## 6. Evaluation
Schema validity comes from the pipeline's own validator. The **G-Eval-lite judge** (`cfg.judge_model`)
then reviews every delivered question: it picks the correct option itself (without seeing the marked
answer) and scores relevance and clarity 1-5. `answer_agreement` = share of questions where its pick
equals the generator's `answer_index`. The judge is evaluation-only; the Space never runs it.

In [ ]:
judgments = []
if cfg.judge:
    judge = model_builder.build_judge(cfg, device, reuse=pipe)
    judgments = engine.judge_quizzes(judge, [q for run in runs.values() for q in run["quizzes"]],
                                     max_new_tokens=cfg.judge_max_new_tokens)
    del judge
summary = engine.summarize(runs, judgments)
run_seconds = time.perf_counter() - run_start
pd.DataFrame(summary)

In [ ]:
plots = cfg.out_dir / "assets"
display(Image(filename=str(export.plot_comparison(summary, plots / "variant_comparison.png"))))
judge_png = export.plot_judge(judgments, plots / "judge_scores.png")
if judge_png:
    display(Image(filename=str(judge_png)))
if judgments:
    display(pd.DataFrame(judgments)[["variant", "sample", "difficulty", "question", "answer_index",
                                     "judge_answer", "relevance", "clarity"]])

In [ ]:
# One generated quiz, as the Space's /predict returns it (questions part)
example = next(q for run in runs.values() for q in run["quizzes"] if q["questions"])
print(example["variant"], example["sample"], example["difficulty"], "valid:", example["valid"],
      "| attempts:", example["attempts"], "| parse:", example["parse"])
for i, q in enumerate(example["questions"], 1):
    print(f"Q{i}. {q['question']}")
    for j, opt in enumerate(q["options"]):
        print(f"   {'*' if j == q['answer_index'] else ' '} {'ABCD'[j]}) {opt}")
    print("   ->", q["explanation"])

## 7. Inference
`P.load(device).predict(...)` on the files in `../space/examples/`: exactly what the Space's `/predict`
runs (media file first, pasted transcript otherwise).

In [ ]:
if not (pipe.asr_model_id == P.ASR_MODELS[P._kind(device)] and pipe.llm_model_id == P.LLM_MODELS[P._kind(device)]):
    del pipe
    model_builder.release()
    pipe = P.load(device)
examples = utils.SPACE_DIR / "examples"
t0 = time.perf_counter()
quiz = pipe.predict(media=examples / "sample.mp4", n_questions=cfg.n_questions, difficulty="easy")
print(f"video -> quiz in {time.perf_counter() - t0:.1f} s | valid={quiz['valid']} | attempts={quiz['attempts']} | transcript: {quiz['transcript']!r}")
display(JSON(quiz))

In [ ]:
if not cfg.smoke:   # the text path (skipped in the smoke run to keep it short)
    text = (examples / "gettysburg.txt").read_text(encoding="utf-8")
    quiz_text = pipe.predict(media=None, transcript=text, n_questions=cfg.n_questions, difficulty="medium")
    display(JSON(quiz_text))

## 8. Export
Writes `metrics.json` (ml-lab schema; headline = the deployed ZeroGPU variant), every quiz with the raw
model replies, the judgments and the plots to `cfg.out_dir` (`outputs/results`, or `outputs/smoke`),
and refreshes the results table in `../model/README.md` (a copy in `outputs/smoke/` for smoke runs).

In [ ]:
metrics = export.build_metrics(cfg, summary, runs, params, run_seconds, device)
out_dir = export.export(cfg, runs, judgments, metrics)
print("exported to", out_dir.relative_to(utils.PROJECT_DIR))
for p in sorted(out_dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(out_dir)}  ({p.stat().st_size / 1024:.1f} KB)")
metrics["primary_metric"], metrics["metrics"]